# Stage 1 — Dataset Generation for RAG

## Project Overview

This notebook implements the first stage of the project: the automatic generation of an instruction-response dataset from a domain-specific document.

The knowledge source used in this work is the official user manual of the **Midea MFM01D110WB 11kg Washing Machine**. The generated dataset will be used in subsequent stages for Retrieval-Augmented Generation (RAG), LoRA-based fine-tuning, model evaluation, and RESTful API deployment.

---

## Objectives

* Extract textual content from the PDF manual.
* Split the document into manageable chunks.
* Generate instruction-response pairs using a local Large Language Model (LLM).
* Validate and curate the generated examples.
* Export the final dataset in JSONL format for fine-tuning.

## Knowledge Source

**Document:** Midea MFM01D110WB 11kg Washing Machine User Manual

The manual contains operational instructions, installation guidelines, safety recommendations, maintenance procedures, troubleshooting information, and technical specifications. These characteristics make it an appropriate domain-specific knowledge source for instruction tuning and question-answering tasks.

## Language Model

The instruction-response pairs are generated using:

* **Model:** Qwen/Qwen2.5-1.5B-Instruct
* **Parameters:** 1.5 Billion
* **Task:** Instruction-response generation

## Expected Output

The final dataset must follow the JSONL format below:

```json
{
  "Instruction": "Example question",
  "Output": "Example answer"
}
```

The resulting dataset will serve as the training corpus for the LoRA fine-tuning stage.

---

## 1. Installing Dependencies

This section installs the libraries required for document processing, dataset generation, language model inference, and data manipulation.

The dependencies include:

* **pdfplumber** for PDF text extraction.
* **transformers** for loading and running Large Language Models (LLMs).
* **torch** as the deep learning framework.
* **accelerate** for optimized model execution.
* **tqdm** for progress monitoring during dataset generation.

These libraries provide the foundation for the Retrieval-Augmented Generation (RAG) dataset creation pipeline implemented in this notebook.


In [1]:
%pip install transformers torch accelerate pdfplumber tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Importing Libraries
This section imports the Python libraries required for PDF processing, dataset generation, model inference, and file manipulation.

In [2]:
import json
import re

import pdfplumber
import torch

from tqdm import tqdm
from transformers import pipeline, logging

# Display only critical Transformers messages
logging.set_verbosity_error()

c:\Users\analu\Documents\Ana Luiza\Documentos - TAIAA\midea-mfm01d110wb-rag-lora-api\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Extracting Text from the Knowledge Source

This section defines the function responsible for extracting textual content from the PDF document that serves as the project's knowledge base.

In [5]:
import os

print(os.getcwd())

c:\Users\analu\Documents\Ana Luiza\Documentos - TAIAA\midea-mfm01d110wb-rag-lora-api\notebooks


In [8]:
def extract_text_from_file(file_path):
    """
    Extracts text from a PDF or TXT file.

    Parameters
    ----------
    file_path : str
        Path to the input file.

    Returns
    -------
    str
        Complete extracted text content.
    """

    if file_path.lower().endswith(".pdf"):
        text = ""

        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()

                if page_text:
                    text += page_text + "\n"

        return text

    elif file_path.lower().endswith(".txt"):

        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    else:
        raise ValueError(
            "Unsupported file format. Please use PDF or TXT files."
        )
    
PDF_PATH = "../data/pdf/manual.pdf"

manual_text = extract_text_from_file(PDF_PATH)

print(f"Total characters extracted: {len(manual_text):,}")

print("\nText preview:\n")
print(manual_text[:1000])

Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported


Total characters extracted: 81,898

Text preview:

LAVA E SECA
MANUAL DO USUÁRIO
11kg
MODELOS:
MFM01D110WB
www.midea.com/br
Obrigado por escolher a Midea!
A Midea é uma empresa comprometida com o bem-estar das pessoas. Com a
combinação de design inteligente e tecnologia, seu novo equipamento trará
novas experiências e deixará seu dia a dia muito mais agradável. Uma receita
simples que fez da Midea uma das maiores fabricantes de eletrodomésticos e
condicionadores de ar do mundo.
Este manual foi feito especialmente para que você conheça todas as características
do seu aparelho, além de informações sobre manutenção, execução de serviços
e claro, como obter o máximo das suas funcionalidades.
Caso precise de informações adicionais ou tenha dúvidas sobre a garantia,
entre em contato através do nosso Serviço de Atendimento ao Consumidor,
pelos telefones ou pelo site abaixo.
SAC - Serviço de Atendimento ao Consumidor
3003 1005 (capitais e regiões metropolitanas)
0800 648 1005 (demais localidad

### Saving Extracted Text

In [12]:
with open(
    "../data/processed/manual_extracted.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(manual_text)

print("Extracted text saved successfully.")

Extracted text saved successfully.
